# Imports

In [1]:
import pandas as pd
import numpy as np

# 🧪 Data generation

This notebook focuses on generating a realistic synthetic retail dataset
(clients, stores, products, transactions) to be used in downstream analytics
and data pipeline workflows.

---

## 🏗️ Dataset Structure

We simulate a realistic retail environment with four core tables:

- **clients**
- **stores**
- **products**
- **transactions**

---

## 📦 What the data includes

- Geographic information *(city, region)*
- Product hierarchy and pricing
- Customer purchase behavior over time
- Time-based purchasing patterns *(daily and hourly behavior)*
- Multiple products per transaction *(basket simulation)*

--- 

## 🔄 Pipeline Usage

This dataset flows through the following pipeline:

1. **Storage** → AWS S3 *(data lake)*
2. **Transformation** → AWS Glue *(ETL, Spark)*
3. **Querying** → AWS Athena *(SQL)*
4. **Modeling** → KMeans clustering *(customer segmentation)*

--- 

> 🎯 Goal: generate realistic retail data to enable scalable analytics, ETL processing, and customer segmentation.

## 🛍️ Products

This section generates the product catalog used in the retail dataset.

Each product is defined by:
- a hierarchical category structure
- a brand (including the house brand)
- a generated product name
- a realistic price based on category

The goal is to simulate a diverse and structured product space similar to real retail environments.

In [2]:
# --- Configuration ---

rng = np.random.default_rng(42)

N_PRODUCTS = 10_000
HOUSE_BRAND_NAME = "Kaminski"

# Category hierarchy
CATEGORY_MAPPING = {
    "Food": ["Dairy", "Meat", "Fruits_Vegetables", "Snacks", "Frozen"],
    "Drinks": ["Soft_Drinks", "Alcohol", "Water", "Coffee_Tea"],
    "Household": ["Cleaning", "Paper_Goods"],
    "Personal_Care": ["Hygiene", "Cosmetics"]
}

# Brand distribution
BRANDS_BY_CATEGORY_LEVEL_1 = {
    "Dairy": ["Danone", "Nestlé", "Arla"],
    "Meat": ["PrimeButcher", "FineCuts", "FarmHouse"],
    "Fruits_Vegetables": ["FreshFields", "GreenFarm", "LocalHarvest"],
    "Snacks": ["Nestlé", "Lay's", "Haribo"],
    "Frozen": ["Iglo", "Findus", "Nestlé"],
    "Soft_Drinks": ["Coca-Cola", "Sprite", "Schweppes"],
    "Alcohol": ["AB InBev", "Heineken", "Carlsberg"],
    "Water": ["Spa", "Vittel", "Perrier"],
    "Coffee_Tea": ["Nescafé", "Douwe Egberts", "Lipton"],
    "Cleaning": ["Unilever", "Henkel", "CleanHome"],
    "Paper_Goods": ["Lotus", "Essity", "SoftCare"],
    "Hygiene": ["Nivea", "Dove", "Garnier"],
    "Cosmetics": ["L'Oréal", "Nivea", "Garnier"]
}

# Product naming vocabulary
PRODUCT_WORDS = {
    "Dairy": ["Yogurt", "Milk", "Cheese", "Butter"],
    "Meat": ["Chicken", "Beef", "Ham", "Sausage"],
    "Fruits_Vegetables": ["Apples", "Bananas", "Tomatoes", "Carrots"],
    "Snacks": ["Chips", "Cookies", "Crackers", "Chocolate"],
    "Frozen": ["Pizza", "Fries", "Vegetables", "IceCream"],
    "Soft_Drinks": ["Cola", "Lemonade", "IcedTea", "Soda"],
    "Alcohol": ["Beer", "Wine", "Vodka", "Cider"],
    "Water": ["StillWater", "SparklingWater"],
    "Coffee_Tea": ["Coffee", "Tea", "Espresso", "GreenTea"],
    "Cleaning": ["Detergent", "Bleach", "DishSoap", "Cleaner"],
    "Paper_Goods": ["ToiletPaper", "PaperTowels", "Napkins"],
    "Hygiene": ["Shampoo", "Soap", "Toothpaste", "Deodorant"],
    "Cosmetics": ["Cream", "Mascara", "Lipstick", "Lotion"]
}

# Price ranges per category
PRICE_RANGES = {
    "Dairy": (1.0, 6.0),
    "Meat": (3.0, 15.0),
    "Fruits_Vegetables": (0.8, 8.0),
    "Snacks": (1.0, 5.0),
    "Frozen": (2.0, 10.0),
    "Soft_Drinks": (1.0, 4.0),
    "Alcohol": (3.0, 20.0),
    "Water": (0.5, 3.0),
    "Coffee_Tea": (2.0, 12.0),
    "Cleaning": (2.0, 12.0),
    "Paper_Goods": (1.5, 10.0),
    "Hygiene": (2.0, 10.0),
    "Cosmetics": (4.0, 25.0)
}

⚙️ Generation Logic

Products are generated by:
- selecting a category hierarchy
- assigning a brand (including house brand probability)
- generating a price based on category ranges
- constructing a product name using brand + keywords

In [3]:
def pick_category_level_0(rng):
    choices = ["Food", "Drinks", "Household", "Personal_Care"]
    probs = [0.45, 0.20, 0.20, 0.15]
    return rng.choice(choices, p=probs)

def pick_category_level_1(category_level_0, rng):
    return rng.choice(CATEGORY_MAPPING[category_level_0])

def pick_house_brand(rng):
    return rng.choice([True, False], p=[0.35, 0.65])

def pick_brand(category_level_1, house_brand, rng):
    if house_brand:
        return HOUSE_BRAND_NAME
    return rng.choice(BRANDS_BY_CATEGORY_LEVEL_1[category_level_1])

def generate_price(category_level_1, house_brand, rng):
    low, high = PRICE_RANGES[category_level_1]
    price = rng.uniform(low, high)

    if house_brand:
        price *= rng.uniform(0.70, 0.90)
    else:
        price *= rng.uniform(0.95, 1.10)

    return round(max(price, 0.5), 2)

def generate_product_name(brand, category_level_1, product_id, rng):
    word = rng.choice(PRODUCT_WORDS[category_level_1])
    return f"{brand} {word} {product_id}"

In [4]:
rows = []

for product_id in range(1, N_PRODUCTS + 1):
    cat0 = pick_category_level_0(rng)
    cat1 = pick_category_level_1(cat0, rng)
    house_brand = pick_house_brand(rng)
    brand = pick_brand(cat1, house_brand, rng)
    unit_price = generate_price(cat1, house_brand, rng)
    product_name = generate_product_name(brand, cat1, product_id, rng)

    rows.append({
        "product_id": product_id,
        "product_name": product_name,
        "category_level_0": cat0,
        "category_level_1": cat1,
        "brand": brand,
        "house_brand": house_brand,
        "unit_price": unit_price
    })

products = pd.DataFrame(rows)
products.head()

,product_id,product_name,category_level_0,category_level_1,brand,house_brand,unit_price
0,1,Essity PaperTowels 1,Household,Paper_Goods,Essity,False,7.16
1,2,Essity ToiletPaper 2,Household,Paper_Goods,Essity,False,5.36
2,3,Nivea Cream 3,Personal_Care,Cosmetics,Nivea,False,13.10
3,4,FreshFields Tomatoes 4,Food,Fruits_Vegetables,FreshFields,False,6.28
4,5,Garnier Soap 5,Personal_Care,Hygiene,Garnier,False,3.63


In [5]:
products = pd.DataFrame(rows)

products.head()

,product_id,product_name,category_level_0,category_level_1,brand,house_brand,unit_price
0,1,Essity PaperTowels 1,Household,Paper_Goods,Essity,False,7.16
1,2,Essity ToiletPaper 2,Household,Paper_Goods,Essity,False,5.36
2,3,Nivea Cream 3,Personal_Care,Cosmetics,Nivea,False,13.10
3,4,FreshFields Tomatoes 4,Food,Fruits_Vegetables,FreshFields,False,6.28
4,5,Garnier Soap 5,Personal_Care,Hygiene,Garnier,False,3.63


In [6]:
products["category_level_0"].value_counts()

category_level_0
Food             4545
Household        2012
Drinks           1969
Personal_Care    1474
Name: count, dtype: int64

Raw data is stored as .csv and transformed into .parquet for analytics.

In [7]:
products.to_csv("products.csv", index=False)
products.to_parquet("products.parquet", index=False)

## 🏪 Stores

This section defines the store network used in the retail dataset.

Each store is associated with:
- a city
- a province
- a region (Flanders, Wallonia, Brussels)

The goal is to simulate a geographically distributed retail network with realistic store density across major Belgian cities.

In [8]:
# --- Store locations ---

STORES_DATA = [
    {"city": "Brussels", "province": "Brussels-Capital", "region": "Brussels"},
    {"city": "Antwerp", "province": "Antwerp", "region": "Flanders"},
    {"city": "Ghent", "province": "East Flanders", "region": "Flanders"},
    {"city": "Bruges", "province": "West Flanders", "region": "Flanders"},
    {"city": "Leuven", "province": "Flemish Brabant", "region": "Flanders"},
    {"city": "Hasselt", "province": "Limburg", "region": "Flanders"},
    {"city": "Mechelen", "province": "Antwerp", "region": "Flanders"},
    {"city": "Aalst", "province": "East Flanders", "region": "Flanders"},
    {"city": "Kortrijk", "province": "West Flanders", "region": "Flanders"},
    {"city": "Genk", "province": "Limburg", "region": "Flanders"},
    {"city": "Liege", "province": "Liege", "region": "Wallonia"},
    {"city": "Namur", "province": "Namur", "region": "Wallonia"},
    {"city": "Charleroi", "province": "Hainaut", "region": "Wallonia"},
    {"city": "Mons", "province": "Hainaut", "region": "Wallonia"},
    {"city": "Tournai", "province": "Hainaut", "region": "Wallonia"},
    {"city": "Arlon", "province": "Luxembourg", "region": "Wallonia"},
    {"city": "La Louviere", "province": "Hainaut", "region": "Wallonia"},
    {"city": "Verviers", "province": "Liege", "region": "Wallonia"},
    {"city": "Wavre", "province": "Walloon Brabant", "region": "Wallonia"},
    {"city": "Nivelles", "province": "Walloon Brabant", "region": "Wallonia"}
]

In [9]:
stores = pd.DataFrame(STORES_DATA)

### 📍 Store Distribution

Stores are not uniformly distributed across cities.

Instead, larger cities (e.g., Brussels, Antwerp) are assigned higher weights to reflect:
- higher population density
- higher store concentration

These weights are later used during transaction generation to simulate realistic shopping behavior.

In [10]:
CITY_WEIGHTS = {
    "Brussels": 10,
    "Antwerp": 8,
    "Ghent": 7,
    "Charleroi": 6,
    "Liege": 6,
    "Bruges": 5,
    "Namur": 5,
    "Leuven": 5,
    "Mons": 4,
    "Aalst": 4,
    "Mechelen": 4,
    "La Louviere": 3,
    "Kortrijk": 3,
    "Hasselt": 3,
    "Genk": 3,
    "Tournai": 2,
    "Verviers": 2,
    "Wavre": 2,
    "Nivelles": 2,
    "Arlon": 1
}

⚙️ Store Table Construction

We enrich the store dataset by:
- assigning weights based on city importance
- generating unique store identifiers
- creating store names
- selecting relevant columns for downstream use

In [11]:
stores["weight"] = stores["city"].map(CITY_WEIGHTS)

store_weights = stores["weight"] / stores["weight"].sum()

stores["store_id"] = range(1, len(stores) + 1)
stores["store_name"] = stores["city"].apply(lambda c: f"Kaminski {c}")

stores = stores[["store_id", "store_name", "city", "province", "region"]]
stores.head()

,store_id,store_name,city,province,region
0,1,Kaminski Brussels,Brussels,Brussels-Capital,Brussels
1,2,Kaminski Antwerp,Antwerp,Antwerp,Flanders
2,3,Kaminski Ghent,Ghent,East Flanders,Flanders
3,4,Kaminski Bruges,Bruges,West Flanders,Flanders
4,5,Kaminski Leuven,Leuven,Flemish Brabant,Flanders


Raw data is stored as .csv and transformed into .parquet for analytics.

In [12]:
stores.to_csv("stores.csv", index=False)
stores.to_parquet("stores.parquet", index=False)

## 👥 Clients

This section generates synthetic clients using weighted sampling to approximate demographic and geographic variation across Belgian cities.

Each client is assigned:
- a gender
- an age group
- a city, province, and region

Geographic assignment is based on store-related city weights to reflect a more realistic population distribution.

In [13]:
# --- Client generation parameters ---

N_CLIENTS = 100_000
rng = np.random.default_rng(42)

GENDERS = ["Male", "Female"]
GENDER_PROBS = [0.5, 0.5]

AGE_GROUPS = ["18_24", "25_34", "35_44", "45_54", "55_plus"]
AGE_PROBS = [0.15, 0.25, 0.25, 0.20, 0.15]

⚙️ Client Generation Logic

Clients are generated by sampling:
- gender
- age group
- geographic location

City assignment is weighted using the store distribution defined earlier, so larger cities are more likely to receive more clients.

In [14]:
rows = []

for client_id in range(1, N_CLIENTS + 1):
    gender = rng.choice(GENDERS, p=GENDER_PROBS)
    age_group = rng.choice(AGE_GROUPS, p=AGE_PROBS)

    idx = rng.choice(stores.index, p=store_weights)
    store_row = stores.loc[idx]

    rows.append({
        "client_id": client_id,
        "gender": gender,
        "age_group": age_group,
        "city": store_row["city"],
        "province": store_row["province"],
        "region": store_row["region"]
    })

clients = pd.DataFrame(rows)
clients.head()

,client_id,gender,age_group,city,province,region
0,1,Female,35_44,Mons,Hainaut,Wallonia
1,2,Female,18_24,Wavre,Walloon Brabant,Wallonia
2,3,Female,45_54,Antwerp,Antwerp,Flanders
3,4,Male,25_34,La Louviere,Hainaut,Wallonia
4,5,Female,45_54,Hasselt,Limburg,Flanders


In [15]:
print(clients.shape)
print("\nGender distribution:")
print(clients["gender"].value_counts(normalize=True).round(3))

print("\nAge group distribution:")
print(clients["age_group"].value_counts(normalize=True).round(3))

print("\nTop 10 cities:")
print(clients["city"].value_counts().head(10))

(100000, 6)

Gender distribution:
gender
Male      0.501
Female    0.499
Name: proportion, dtype: float64

Age group distribution:
age_group
35_44      0.252
25_34      0.251
45_54      0.198
55_plus    0.150
18_24      0.149
Name: proportion, dtype: float64

Top 10 cities:
city
Brussels     11717
Antwerp       9438
Ghent         8320
Charleroi     7168
Liege         7145
Namur         5854
Leuven        5805
Bruges        5770
Mons          4731
Mechelen      4668
Name: count, dtype: int64


Raw data is stored as .csv and transformed into .parquet for analytics.

In [16]:
clients.to_csv("clients.csv", index=False)
clients.to_parquet("clients.parquet", index=False)

## 🧾 Transactions

This section generates transaction-level purchase events linking clients, stores, and products.

The transaction logic is designed to mimic realistic retail behavior by modeling:
- basket sizes
- product popularity
- category-based co-purchases
- store selection tied to client location
- time-based purchasing patterns

In [17]:
# --- Transaction generation parameters ---
N_BASKETS = 200_000
rng = np.random.default_rng(42)

In [18]:
# --- Helper mappings ---
product_prices = products.set_index("product_id")["unit_price"].to_dict()
client_ids = clients["client_id"].values

client_location = clients.set_index("client_id")[["city", "province", "region"]]
city_to_store_ids = stores.groupby("city")["store_id"].apply(list).to_dict()
all_store_ids = stores["store_id"].tolist()

⚖️ Product Popularity Modeling

Products are not sampled uniformly.

To reflect more realistic purchasing behavior, category-level purchase weights are combined with product-specific random popularity factors. This makes some categories and products more likely to appear in baskets than others.

In [19]:
# --- Products popularity weights ---

CATEGORY_PURCHASE_WEIGHTS = {
    "Dairy": 1.8,
    "Meat": 1.4,
    "Fruits_Vegetables": 1.7,
    "Snacks": 1.6,
    "Frozen": 1.2,
    "Soft_Drinks": 1.5,
    "Alcohol": 0.7,
    "Water": 1.3,
    "Coffee_Tea": 1.0,
    "Cleaning": 0.6,
    "Paper_Goods": 0.8,
    "Hygiene": 0.7,
    "Cosmetics": 0.4
}

In [20]:
products_for_transactions = products.copy()
products_for_transactions["category_weight"] = (
    products_for_transactions["category_level_1"].map(CATEGORY_PURCHASE_WEIGHTS)
)
products_for_transactions["product_popularity"] = rng.lognormal(
    mean=0.0,
    sigma=0.8,
    size=len(products_for_transactions)
)
products_for_transactions["purchase_weight"] = (
    products_for_transactions["category_weight"]
    * products_for_transactions["product_popularity"]
)

# Normalizing weights into sampling probabilities
product_purchase_probs = (
    products_for_transactions["purchase_weight"]
    / products_for_transactions["purchase_weight"].sum()
).values

product_ids = products_for_transactions["product_id"].values

product_to_category_level_0 = (
    products_for_transactions.set_index("product_id")["category_level_0"].to_dict()
)
products_by_category_level_0 = (
    products_for_transactions.groupby("category_level_0")["product_id"].apply(list).to_dict()
)

🕒 Datetime Modeling

Transaction timestamps are generated across a defined date range, with different purchase time patterns for weekdays and weekends.

In [21]:
# --- Date range for transactions ---
start_date = pd.Timestamp("2025-01-01")
end_date = pd.Timestamp("2025-12-31")

In [22]:
def generate_realistic_datetime(rng, start_date, end_date):
    days = (end_date - start_date).days + 1
    random_day = start_date + pd.Timedelta(days=int(rng.integers(0, days)))
    weekday = random_day.weekday()

    # Week-end (Fri, Sat, Sun):
    if weekday >= 4: 
        period = rng.choice(
            ["morning", "lunch", "evening", "off"],
            p=[0.25, 0.25, 0.40, 0.10]
        )
    else:
        period = rng.choice(
            ["morning", "lunch", "evening", "off"],
            p=[0.30, 0.30, 0.30, 0.10]
        )

    if period == "morning":
        hour = rng.integers(8, 12)
    elif period == "lunch":
        hour = rng.integers(12, 15)
    elif period == "evening":
        hour = rng.integers(17, 21)
    else:
        hour = rng.integers(6, 23)

    minute = rng.integers(0, 60)
    second = rng.integers(0, 60)

    return pd.Timestamp(
        year=random_day.year,
        month=random_day.month,
        day=random_day.day,
        hour=int(hour),
        minute=int(minute),
        second=int(second)
    )


🛒 Basket Generation Logic

Each basket is generated by:
- sampling a client
- assigning a likely store based on client location
- generating a realistic timestamp
- selecting an anchor product based on popularity
- filling the basket with a mix of related and general products
- assigning quantities and computing total amounts

In [23]:
rows = []

for transaction_id in range(1, N_BASKETS + 1):
    # Choosing a client
    client_id = rng.choice(client_ids)
    client_city = client_location.loc[client_id, "city"]

    # Choosing a store
    if client_city in city_to_store_ids and rng.random() < 0.85:
        store_id = rng.choice(city_to_store_ids[client_city])
    else:
        store_id = rng.choice(all_store_ids)

    # Choosing datetime
    transaction_datetime = generate_realistic_datetime(rng, start_date, end_date)

    # Size of the basket
    basket_size = rng.choice(
        [1, 2, 3, 4, 5, 6, 7],
        p=[0.10, 0.18, 0.22, 0.20, 0.14, 0.10, 0.06]
    )

    # Choosing anchor product using popularity weights
    anchor_product = rng.choice(product_ids, p=product_purchase_probs)
    anchor_cat0 = product_to_category_level_0[anchor_product]

    basket_product_ids = {anchor_product}

    while len(basket_product_ids) < basket_size:
        # 70% chance to stay in same broad category, 30% across all products
        if rng.random() < 0.70:
            candidate = rng.choice(products_by_category_level_0[anchor_cat0])
        else:
            candidate = rng.choice(product_ids, p=product_purchase_probs)

        basket_product_ids.add(candidate)

    # Generating rows
    for product_id in basket_product_ids:
        quantity = rng.choice([1, 2, 3, 4, 5], p=[0.60, 0.22, 0.10, 0.05, 0.03])
        unit_price = product_prices[product_id]
        total_amount = round(quantity * unit_price, 2)

        rows.append({
            "transaction_id": transaction_id,
            "transaction_datetime": transaction_datetime,
            "client_id": client_id,
            "store_id": store_id,
            "product_id": product_id,
            "quantity": int(quantity),
            "unit_price": float(unit_price),
            "total_amount": total_amount
        })

transactions = pd.DataFrame(rows)

In [24]:
print(f"Transactions table shape: {transactions.shape}")
print(f"Unique baskets: {transactions['transaction_id'].nunique()}")

print("\nBasket size summary:")
print(transactions.groupby("transaction_id").size().describe().round(2))

print("\nQuantity distribution:")
print(transactions["quantity"].value_counts(normalize=True).sort_index().round(3))

Transactions table shape: (729194, 8)
Unique baskets: 200000

Basket size summary:
count    200000.00
mean          3.65
std           1.67
min           1.00
25%           2.00
50%           4.00
75%           5.00
max           7.00
dtype: float64

Quantity distribution:
quantity
1    0.60
2    0.22
3    0.10
4    0.05
5    0.03
Name: proportion, dtype: float64


Raw data is stored as .csv and transformed into .parquet for analytics.

In [25]:
transactions.to_parquet("transactions.parquet", index=False)
transactions.to_csv("transactions.csv", index=False)

## 📤 Sample exports for GitHub preview

In [26]:
import os
os.makedirs("data_sample", exist_ok=True)


products.sample(100, random_state=42)\
    .to_csv("data_sample/products_sample.csv", index=False)

clients.sample(100, random_state=42)\
    .to_csv("data_sample/clients_sample.csv", index=False)

transactions.sample(500, random_state=42)\
    .to_csv("data_sample/transactions_sample.csv", index=False)

stores.to_csv("data_sample/stores.csv", index=False)